# La Ronde Fireworks — Multi-Year Training & Model Export

This notebook **trains** the fireworks detector on every complete BIXI season we have a
confirmed IFLQ calendar for (**2022–2025**) and **saves the fitted models** to
`output/firework_model.joblib`. Once saved, `predict_fireworks.py` just loads that artifact,
runs the shared preprocessing on a new season's CSV, and scores it — no retraining.

**No train/serve skew:** all feature engineering lives in `firework_features.py`, imported
here *and* by the predictor, so the two can never drift apart.

Pipeline: coordinate-matched viewshed → nightly pre-show/escape windows → per-year-portable
features (within-night ratios, weekday, weather — never raw counts) → labels from
`fireworks_dates.json` → **leave-one-year-out** evaluation (the honest cross-year test) →
fit on all years → export bundle.

In [1]:
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

import firework_features as ff   # shared preprocessing (train == predict)

DATA_DIR = Path('/Volumes/Extreme SSD/SUMO Data')
TRAIN_CSVS = {
    2022: DATA_DIR / 'DonneesOuverte2022.csv',
    2023: DATA_DIR / 'DonneesOuvertes2023_12.csv',
    2024: DATA_DIR / 'DonneesOuvertes2024_010203040506070809101112.csv',
    2025: DATA_DIR / 'DonneesOuvertes2025_010203040506070809101112.csv',
}
MODEL_OUT   = Path('output/firework_model.joblib')
MODEL_OUT.parent.mkdir(exist_ok=True)
USE_WEATHER = True   # Open-Meteo archive covers all past years
print('sklearn', sklearn.__version__)

sklearn 1.9.0


## 1. Load every training season and label it

Each CSV is streamed through the shared `build_features`: keep trips touching the viewshed
box (matched on the trip's own station coordinates, so a changing station network doesn't
move the zone), aggregate into nightly pre-show / escape windows, and derive the portable
features. Labels come straight from `fireworks_dates.json` — the count printed per year is a
built-in check that every confirmed show landed in the data.

In [2]:
fw_dates = ff.load_fireworks_dates()          # {year: {date, ...}} from the JSON DB
print('label years:', sorted(fw_dates), '\n')

frames, weather_all = [], True
for yr, path in TRAIN_CSVS.items():
    print(f'Loading {yr}: {path.name}')
    n, wok = ff.build_features(str(path), use_weather=USE_WEATHER)
    if n is None:
        print('  [skip] no June-August nights in the viewshed'); continue
    n = n.copy()
    n['is_fireworks'] = ff.label_series(n.index, fw_dates.get(yr, set()))
    n['year'] = yr
    weather_all &= wok
    got, exp = int(n['is_fireworks'].sum()), len(fw_dates.get(yr, []))
    flag = 'OK' if got == exp else f'!! MISMATCH (expected {exp})'
    print(f'  {len(n):3d} JJA nights | {got} fireworks labelled  {flag}')
    frames.append(n)

data = pd.concat(frames).sort_index()
print(f'\nPooled: {len(data)} nights across {data.year.nunique()} years, '
      f"{int(data['is_fireworks'].sum())} fireworks nights")

label years: [2022, 2023, 2024, 2025, 2026] 

Loading 2022: DonneesOuverte2022.csv
   92 JJA nights | 9 fireworks labelled  OK
Loading 2023: DonneesOuvertes2023_12.csv
   92 JJA nights | 8 fireworks labelled  OK
Loading 2024: DonneesOuvertes2024_010203040506070809101112.csv
   92 JJA nights | 8 fireworks labelled  OK
Loading 2025: DonneesOuvertes2025_010203040506070809101112.csv
   92 JJA nights | 8 fireworks labelled  OK

Pooled: 368 nights across 4 years, 33 fireworks nights


## 2. Feature matrix

Weather features are included only if every training year fetched them (otherwise the
columns would be ragged). The exact `FEATURES` list is saved with the model so the predictor
builds an identically-shaped matrix.

In [3]:
USE_WEATHER = USE_WEATHER and weather_all
FEATURES = ff.feature_list(USE_WEATHER)
data[FEATURES] = data[FEATURES].fillna(0.0)

X = data[FEATURES]
y = data['is_fireworks'].astype(int)
groups = data['year']
print(f'{len(X)} nights x {len(FEATURES)} features | {int(y.sum())} positives '
      f'(base rate {y.mean():.3f})')
print('features:', FEATURES)
X.describe().T[['mean', 'min', 'max']]

368 nights x 6 features | 33 positives (base rate 0.090)
features: ['escape_share', 'netout_share', 'pre_arrival_share', 'dow', 'temp_max', 'precip_mm']


,mean,min,max
escape_share,0.402835,0.155313,0.703660
netout_share,0.077685,-0.164993,0.593848
pre_arrival_share,0.376797,0.191263,0.559491
dow,3.010870,0.000000,6.000000
temp_max,25.481250,13.300000,34.800000
precip_mm,4.355163,0.000000,98.500000


## 3. Leave-one-year-out evaluation

The decisive test: predict each season with a model that trained **only on the other years**.
This is genuinely out-of-sample (a whole season held out at a time), unlike within-year
cross-validation. With the years spanning Wed/Sat (2022) through Thu/Sun (2025) schedules, a
model that scores well here is reading the *demand shape*, not memorising a weekday.

In [4]:
logo = LeaveOneGroupOut()

def make_logreg(C=0.3):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, max_iter=5000, class_weight='balanced'))

MODELS = {
    'logreg': make_logreg(0.3),
    'gb': GradientBoostingClassifier(n_estimators=300, max_depth=2,
                                     learning_rate=0.03, random_state=42),
}

loyo_proba, loyo_scores = {}, {}
for name, mdl in MODELS.items():
    p = cross_val_predict(mdl, X, y, cv=logo, groups=groups, method='predict_proba')[:, 1]
    loyo_proba[name] = p
    loyo_scores[name] = {'roc': float(roc_auc_score(y, p)),
                         'ap': float(average_precision_score(y, p))}
    print(f"{name:8s}  ROC-AUC {loyo_scores[name]['roc']:.3f}  "
          f"PR-AUC {loyo_scores[name]['ap']:.3f}")

print('\nPer-year recall @ 0.5 (leave-one-year-out, logreg):')
for yr in sorted(groups.unique()):
    m = (groups == yr).to_numpy()
    called = loyo_proba['logreg'][m] >= 0.5
    truth = y.to_numpy()[m].astype(bool)
    tp = int((called & truth).sum())
    print(f'  {yr}: caught {tp}/{int(truth.sum())} shows, '
          f'{int((called & ~truth).sum())} false alarms')

logreg    ROC-AUC 0.978  PR-AUC 0.954
gb        ROC-AUC 0.914  PR-AUC 0.855

Per-year recall @ 0.5 (leave-one-year-out, logreg):
  2022: caught 9/9 shows, 2 false alarms
  2023: caught 6/8 shows, 1 false alarms
  2024: caught 8/8 shows, 2 false alarms
  2025: caught 8/8 shows, 2 false alarms


## 4. Operating threshold

Pick the probability cut that maximises F1 on the out-of-fold (leave-one-year-out)
predictions of the default model, then report the confusion counts it implies. This threshold
is saved with the model as the default `--threshold` for the predictor.

The result shows an accuracy of 98.91% with precision and recall both at 94%.

In [5]:
DEFAULT_MODEL = 'logreg'
p = loyo_proba[DEFAULT_MODEL]
prec, rec, thr = precision_recall_curve(y, p)
f1 = np.where((prec[:-1] + rec[:-1]) > 0,
              2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12), 0.0)
i = int(np.argmax(f1))
THRESHOLD = float(thr[i])
called = p >= THRESHOLD
tp = int((called & (y == 1)).sum()); fp = int((called & (y == 0)).sum())
fn = int((~called & (y == 1)).sum())
print(f'Threshold {THRESHOLD:.3f}  precision {prec[i]:.2f}  recall {rec[i]:.2f}  '
      f'F1 {f1[i]:.2f}  (TP={tp} FP={fp} FN={fn})')

Threshold 0.784  precision 0.94  recall 0.94  F1 0.94  (TP=31 FP=2 FN=2)


## 4b. Diagnosing the mistakes

Before touching hyperparameters, look at *which* nights the model gets wrong at the operating
threshold and *why*. A miss on a rainy night (high `precip_mm`, a suppressed exodus) is a data
limitation no tuning can fix; a false alarm on a busy non-show night may be another riverfront
event (Osheaga / ÎleSoniq / Piknic) worth adding as a label. Compare each error's exodus
features (`netout_share`, `escape_netout`) against the median of the shows we caught.

In [ ]:
# Mistakes at the operating threshold, from the leave-one-year-out probabilities.
diag = data.copy()
diag['p'] = loyo_proba[DEFAULT_MODEL]
diag['weekday'] = diag.index.day_name().str[:3]
truth  = diag['is_fireworks'].astype(bool)
called = diag['p'] >= THRESHOLD

cols = ['year', 'weekday', 'p', 'escape_netout', 'escape_rides',
        'escape_share', 'netout_share', 'pre_arrival_share', 'temp_max', 'precip_mm']
cols = [c for c in cols if c in diag.columns]   # weather may be absent if the fetch failed
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

misses       = diag[truth & ~called].sort_values('p', ascending=False)
false_alarms = diag[~truth & called].sort_values('p', ascending=False)
fmt = lambda v: f'{v:.2f}'

print(f'FALSE NEGATIVES - real shows MISSED (p < {THRESHOLD:.2f}):  {len(misses)}')
print(misses[cols].to_string(float_format=fmt) if len(misses) else '  (none)')
print(f'\nFALSE POSITIVES - non-shows FLAGGED (p >= {THRESHOLD:.2f}):  {len(false_alarms)}')
print(false_alarms[cols].to_string(float_format=fmt) if len(false_alarms) else '  (none)')

# Reference: what a CAUGHT show typically looks like, to judge how far each error sits.
hits = diag[truth & called]
ref = ['escape_netout', 'netout_share', 'escape_share', 'precip_mm']
ref = [c for c in ref if c in diag.columns]
print(f'\nReference - median of CAUGHT shows (n={len(hits)}):')
print(hits[ref].median().round(2).to_string())

## 5. Fit on all years and export the bundle

Refit both models on the **full** 2022–2025 pool (no data held back — the honest performance
estimate already came from section 3), then serialise everything the predictor needs: the
fitted pipelines, the exact feature list, the weather flag, the tuned threshold, and
provenance (training years, sklearn version, timestamp). One file, `output/firework_model.joblib`.

In [6]:
final_models = {name: clone(mdl).fit(X, y) for name, mdl in MODELS.items()}

bundle = {
    'models': final_models,
    'features': FEATURES,
    'use_weather': USE_WEATHER,
    'threshold': THRESHOLD,
    'default_model': DEFAULT_MODEL,
    'train_years': sorted(TRAIN_CSVS),
    'loyo_scores': loyo_scores,
    'sklearn_version': sklearn.__version__,
    'trained_at': dt.datetime.now().isoformat(timespec='seconds'),
    'feature_module': 'firework_features',
}
joblib.dump(bundle, MODEL_OUT)
print(f'Saved {MODEL_OUT}  ({MODEL_OUT.stat().st_size/1024:.1f} KB)')
print('  models   :', list(final_models))
print('  features :', FEATURES)
print('  threshold:', round(THRESHOLD, 3), '| default:', DEFAULT_MODEL)

Saved output/firework_model.joblib  (227.0 KB)
  models   : ['logreg', 'gb']
  features : ['escape_share', 'netout_share', 'pre_arrival_share', 'dow', 'temp_max', 'precip_mm']
  threshold: 0.784 | default: logreg


## 6. Which features carry the signal

Standardised logistic-regression coefficients on the full pool. The exodus signal
(`netout_share`, `escape_share`) should dominate; a large `dow` weight would be a warning that the
model leans on the weekday calendar rather than the demand shape.

In [7]:
lr = clone(MODELS['logreg']).fit(X, y)
coef = pd.Series(lr.named_steps['logisticregression'].coef_[0], index=FEATURES)
print(coef.sort_values(ascending=False).round(3).to_string())

netout_share         1.764
pre_arrival_share    1.461
escape_share         1.269
precip_mm            0.298
dow                  0.071
temp_max            -0.091


## 7. Serving — how `predict_fireworks.py` uses this

```bash
python predict_fireworks.py \
    --predict "/Volumes/Extreme SSD/SUMO Data/DonneesOuvertes2026_<full-season>.csv"
```

The predictor loads `output/firework_model.joblib`, runs the **same** `firework_features`
preprocessing on the new CSV, aligns to the saved `features`, and scores every night — no
training. If the predicted season's calendar is in `fireworks_dates.json` it also prints a
validation line against ground truth.

**Retrain only when** the labels change (a new confirmed year) or the feature code changes.
Re-run this notebook top-to-bottom to refresh the artifact.